# Pizza Shop Hourly Order Prediction

Explore hourly order data, train a model, evaluate it on future dates, make predictions, and save the complete pipeline for a later Streamlit app.

## 1. Import the tools

A **pipeline** keeps preprocessing and the model together, so deployment code only needs to call `predict()`.

In [1]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

RANDOM_STATE = 42

## 2. Load and check the data

The dates in this project use ISO format: `YYYY-MM-DD`.

In [2]:
DATA_PATH = Path("data/pizza_orders.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")

print(f"Rows: {len(df):,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

Rows: 4,380
Date range: 2025-01-01 to 2025-12-31


,date,hour,day_of_week,is_weekend,promotion,order_count
0,2025-01-01,11,Wednesday,0,none,12
1,2025-01-01,12,Wednesday,0,none,30
2,2025-01-01,13,Wednesday,0,none,27
3,2025-01-01,14,Wednesday,0,none,19
4,2025-01-01,15,Wednesday,0,none,17


In [3]:
expected_columns = {
    "date", "hour", "day_of_week", "is_weekend",
    "promotion", "order_count"
}
missing_columns = expected_columns - set(df.columns)
assert not missing_columns, f"Missing columns: {sorted(missing_columns)}"

df.info()
display(df.isna().sum().rename("missing_values"))
display(df.describe(include="all"))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4380 entries, 0 to 4379
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   date         4380 non-null   datetime64[ns]
 1   hour         4380 non-null   int64         
 2   day_of_week  4380 non-null   object        
 3   is_weekend   4380 non-null   int64         
 4   promotion    4380 non-null   object        
 5   order_count  4380 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 205.4+ KB


date           0
hour           0
day_of_week    0
is_weekend     0
promotion      0
order_count    0
Name: missing_values, dtype: int64

,date,hour,day_of_week,is_weekend,promotion,order_count
count,4380,4380.000000,4380,4380.000000,4380,4380.000000
unique,NaN,NaN,7,NaN,4,NaN
top,NaN,NaN,Wednesday,NaN,none,NaN
freq,NaN,NaN,636,NaN,4031,NaN
mean,2025-07-02 00:00:00,16.500000,NaN,0.284932,NaN,19.101826
min,2025-01-01 00:00:00,11.000000,NaN,0.000000,NaN,0.000000
25%,2025-04-02 00:00:00,13.750000,NaN,0.000000,NaN,12.000000
50%,2025-07-02 00:00:00,16.500000,NaN,0.000000,NaN,17.000000
75%,2025-10-01 00:00:00,19.250000,NaN,1.000000,NaN,25.000000
max,2025-12-31 00:00:00,22.000000,NaN,1.000000,NaN,63.000000


## 3. Quick exploration

Look for lunch/dinner peaks and whether promotions are associated with more orders. These summaries show patterns, not proof of cause.

In [4]:
hour_summary = (
    df.groupby("hour", as_index=False)["order_count"]
    .mean()
    .rename(columns={"order_count": "average_orders"})
)
promotion_summary = (
    df.groupby("promotion", as_index=False)["order_count"]
    .mean()
    .rename(columns={"order_count": "average_orders"})
    .sort_values("average_orders", ascending=False)
)

display(hour_summary.round(1))
display(promotion_summary.round(1))

,hour,average_orders
0,11,12.3
1,12,21.5
2,13,24.6
3,14,15.6
4,15,10.1
5,16,11.3
6,17,17.4
7,18,25.7
8,19,30.0
9,20,28.1


,promotion,average_orders
0,family_bundle,37.2
1,lunch_special,20.4
3,percent_off,20.1
2,none,18.6


## 4. Prepare features and split chronologically

We create simple calendar features from `date`. We train on the oldest 80% of rows and test on the newest 20%. This better represents predicting future demand than a random split.

In [5]:
model_df = df.sort_values(["date", "hour"]).reset_index(drop=True).copy()
model_df["month"] = model_df["date"].dt.month
model_df["day_of_month"] = model_df["date"].dt.day
model_df["day_of_year"] = model_df["date"].dt.dayofyear

feature_columns = [
    "hour", "day_of_week", "is_weekend", "promotion",
    "month", "day_of_month", "day_of_year"
]
X = model_df[feature_columns]
y = model_df["order_count"]

split_index = int(len(model_df) * 0.80)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")
print(f"Train through: {model_df.loc[split_index - 1, 'date'].date()}")
print(f"Test starts:   {model_df.loc[split_index, 'date'].date()}")

Training rows: 3,504
Test rows:     876
Train through: 2025-10-19
Test starts:   2025-10-20


## 5. Build and train the pipeline

Models need numbers. `OneHotEncoder` turns categories such as `promotion=family_bundle` into numeric indicator columns. `handle_unknown='ignore'` prevents errors if deployment data contains a category not seen during training.

In [6]:
categorical_features = ["day_of_week", "promotion"]
numeric_features = [
    "hour", "is_weekend", "month", "day_of_month", "day_of_year"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("categories", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numbers", "passthrough", numeric_features),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
        )),
    ]
)

pipeline.fit(X_train, y_train)
print("Training complete.")

Training complete.


## 6. Evaluate with MAE

**Mean Absolute Error (MAE)** is the average size of the prediction error. An MAE of 3 means predictions are off by about 3 orders per hour on average. Lower is better.

In [7]:
test_predictions = pipeline.predict(X_test)
mae = mean_absolute_error(y_test, test_predictions)
baseline_predictions = [y_train.mean()] * len(y_test)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"Model MAE:    {mae:.2f} orders")
print(f"Baseline MAE: {baseline_mae:.2f} orders")
print("The baseline always predicts the training-set average.")

Model MAE:    5.80 orders
Baseline MAE: 8.51 orders
The baseline always predicts the training-set average.


In [8]:
results = model_df.iloc[split_index:][["date", "hour", "promotion"]].copy()
results["actual_orders"] = y_test.to_numpy()
results["predicted_orders"] = test_predictions.round(1)
results["absolute_error"] = (
    results["actual_orders"] - results["predicted_orders"]
).abs().round(1)
results.head(10)

,date,hour,promotion,actual_orders,predicted_orders,absolute_error
3504,2025-10-20,11,lunch_special,15,10.4,4.6
3505,2025-10-20,12,none,21,12.2,8.8
3506,2025-10-20,13,none,21,14.0,7.0
3507,2025-10-20,14,none,10,15.8,5.8
3508,2025-10-20,15,none,4,9.4,5.4
3509,2025-10-20,16,none,13,11.2,1.8
3510,2025-10-20,17,none,13,16.1,3.1
3511,2025-10-20,18,percent_off,23,22.8,0.2
3512,2025-10-20,19,none,24,24.9,0.9
3513,2025-10-20,20,none,21,23.4,2.4


## 7. Predict new examples

A later Streamlit app can collect a date, hour, and promotion, calculate the same calendar fields, and pass one row to the saved pipeline.

In [9]:
def make_prediction_input(date, hour, promotion):
    date = pd.Timestamp(date)
    return pd.DataFrame([{
        "hour": hour,
        "day_of_week": date.day_name(),
        "is_weekend": int(date.dayofweek >= 5),
        "promotion": promotion,
        "month": date.month,
        "day_of_month": date.day,
        "day_of_year": date.dayofyear,
    }])

examples = pd.concat([
    make_prediction_input("2026-01-09", 13, "lunch_special"),
    make_prediction_input("2026-01-10", 19, "family_bundle"),
    make_prediction_input("2026-01-12", 16, "none"),
], ignore_index=True)

examples["predicted_orders"] = pipeline.predict(examples).round().astype(int)
examples

,hour,day_of_week,is_weekend,promotion,month,day_of_month,day_of_year,predicted_orders
0,13,Friday,0,lunch_special,1,9,9,29
1,19,Saturday,1,family_bundle,1,10,10,48
2,16,Monday,0,none,1,12,12,14


## 8. Save the complete pipeline

We save the fitted preprocessor and Random Forest together. The Streamlit app can load this one file with `joblib.load()`.

In [10]:
MODEL_PATH = Path("models/pizza_order_pipeline.joblib")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, MODEL_PATH)

loaded_pipeline = joblib.load(MODEL_PATH)
check_prediction = loaded_pipeline.predict(examples.iloc[[0]])[0]

print(f"Saved model to: {MODEL_PATH.resolve()}")
print(f"Reload check: {check_prediction:.1f} predicted orders")

Saved model to: C:\Users\semin\OneDrive\Documents\IIT\guest_lecs\pizza-rush-predictor\models\pizza_order_pipeline.joblib
Reload check: 29.3 predicted orders
